In [1]:
import pandas as pd
import os

# Define paths to all raw CSV files
RAW = "../data/raw/"

files = {
    "orders":       "olist_orders_dataset.csv",
    "items":        "olist_order_items_dataset.csv",
    "payments":     "olist_order_payments_dataset.csv",
    "reviews":      "olist_order_reviews_dataset.csv",
    "customers":    "olist_customers_dataset.csv",
    "products":     "olist_products_dataset.csv",
    "translation":  "product_category_name_translation.csv"
}

# Load all files into a dictionary of DataFrames
dfs = {}
for name, filename in files.items():
    dfs[name] = pd.read_csv(RAW + filename)
    print(f"✅ Loaded {name}: {dfs[name].shape[0]:,} rows × {dfs[name].shape[1]} columns")

✅ Loaded orders: 99,441 rows × 8 columns
✅ Loaded items: 112,650 rows × 7 columns
✅ Loaded payments: 103,886 rows × 5 columns
✅ Loaded reviews: 99,224 rows × 7 columns
✅ Loaded customers: 99,441 rows × 5 columns
✅ Loaded products: 32,951 rows × 9 columns
✅ Loaded translation: 71 rows × 2 columns


In [2]:
def inspect_table(name, df):
    print("=" * 60)
    print(f"TABLE: {name.upper()}")
    print("=" * 60)
    print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
    print("\n--- Column Names & Data Types ---")
    print(df.dtypes)
    print("\n--- Missing Values ---")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print("No missing values ✅")
    else:
        print(nulls)
    print("\n--- Sample Rows (first 3) ---")
    print(df.head(3).to_string())
    print()

for name, df in dfs.items():
    inspect_table(name, df)

TABLE: ORDERS
Rows: 99,441  |  Columns: 8

--- Column Names & Data Types ---
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

--- Missing Values ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- Sample Rows (first 3) ---
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13  

In [3]:
# How many unique orders exist in each table?
print("=== UNIQUE ORDER COUNTS PER TABLE ===")
print(f"orders table:    {dfs['orders']['order_id'].nunique():,} unique orders")
print(f"items table:     {dfs['items']['order_id'].nunique():,} unique orders")
print(f"payments table:  {dfs['payments']['order_id'].nunique():,} unique orders")
print(f"reviews table:   {dfs['reviews']['order_id'].nunique():,} unique orders")

print("\n=== ROWS PER ORDER (checking for one-to-many) ===")
print("Items per order (can be multiple):")
print(dfs['items'].groupby('order_id').size().describe())

print("\nPayments per order (can be multiple):")
print(dfs['payments'].groupby('order_id').size().describe())

print("\nReviews per order (should be 1, check for duplicates):")
print(dfs['reviews'].groupby('order_id').size().describe())

=== UNIQUE ORDER COUNTS PER TABLE ===
orders table:    99,441 unique orders
items table:     98,666 unique orders
payments table:  99,440 unique orders
reviews table:   98,673 unique orders

=== ROWS PER ORDER (checking for one-to-many) ===
Items per order (can be multiple):
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

Payments per order (can be multiple):
count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

Reviews per order (should be 1, check for duplicates):
count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64


In [4]:
print("=== ORDER STATUS DISTRIBUTION ===")
print(dfs['orders']['order_status'].value_counts())
print(f"\nTotal orders: {len(dfs['orders']):,}")
print(f"Delivered orders: {(dfs['orders']['order_status'] == 'delivered').sum():,}")

=== ORDER STATUS DISTRIBUTION ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Total orders: 99,441
Delivered orders: 96,478


In [5]:
print("=== DATE COLUMNS IN ORDERS TABLE ===")
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    null_count = dfs['orders'][col].isnull().sum()
    sample = dfs['orders'][col].dropna().iloc[0]
    print(f"{col}")
    print(f"  Nulls: {null_count:,}  |  Sample value: {sample}")
    print()

=== DATE COLUMNS IN ORDERS TABLE ===
order_purchase_timestamp
  Nulls: 0  |  Sample value: 2017-10-02 10:56:33

order_approved_at
  Nulls: 160  |  Sample value: 2017-10-02 11:07:15

order_delivered_carrier_date
  Nulls: 1,783  |  Sample value: 2017-10-04 19:55:00

order_delivered_customer_date
  Nulls: 2,965  |  Sample value: 2017-10-10 21:25:13

order_estimated_delivery_date
  Nulls: 0  |  Sample value: 2017-10-18 00:00:00



In [6]:
print("=== NUMERIC COLUMN SUMMARIES ===")

print("\n-- Items: price and freight_value --")
print(dfs['items'][['price', 'freight_value']].describe().round(2))

print("\n-- Payments: payment_value and installments --")
print(dfs['payments'][['payment_value', 'payment_installments']].describe().round(2))

print("\n-- Reviews: review_score --")
print(dfs['reviews']['review_score'].value_counts().sort_index())

=== NUMERIC COLUMN SUMMARIES ===

-- Items: price and freight_value --
           price  freight_value
count  112650.00      112650.00
mean      120.65          19.99
std       183.63          15.81
min         0.85           0.00
25%        39.90          13.08
50%        74.99          16.26
75%       134.90          21.15
max      6735.00         409.68

-- Payments: payment_value and installments --
       payment_value  payment_installments
count      103886.00             103886.00
mean          154.10                  2.85
std           217.49                  2.69
min             0.00                  0.00
25%            56.79                  1.00
50%           100.00                  1.00
75%           171.84                  4.00
max         13664.08                 24.00

-- Reviews: review_score --
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [7]:
print("=== PRODUCT CATEGORIES ===")
print(f"Total products: {len(dfs['products']):,}")
print(f"Unique categories (Portuguese): {dfs['products']['product_category_name'].nunique()}")
print(f"Missing category names: {dfs['products']['product_category_name'].isnull().sum()}")

print("\n=== TRANSLATION TABLE ===")
print(f"Categories in translation file: {len(dfs['translation'])}")
print(dfs['translation'].head(10))

=== PRODUCT CATEGORIES ===
Total products: 32,951
Unique categories (Portuguese): 73
Missing category names: 610

=== TRANSLATION TABLE ===
Categories in translation file: 71
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
5           esporte_lazer                sports_leisure
6              perfumaria                     perfumery
7   utilidades_domesticas                    housewares
8               telefonia                     telephony
9      relogios_presentes                 watches_gifts


In [8]:
print("""
=== KEY FINDINGS FROM INSPECTION ===

1. ORDERS TABLE:
   - Contains ~100K orders with multiple date columns
   - order_delivered_customer_date has nulls (non-delivered orders)
   - We will filter to 'delivered' status for delivery analysis

2. ITEMS TABLE:
   - One-to-many with orders (some orders have multiple items)
   - price and freight_value are per item, not per order

3. PAYMENTS TABLE:
   - One-to-many with orders (installments create multiple rows)
   - Must aggregate by order_id before joining to items

4. REVIEWS TABLE:
   - Some orders may have duplicate reviews
   - Will use latest review or average — decide in cleaning step

5. JOIN RISK:
   - payments has multiple rows per order → must aggregate FIRST
   - items has multiple rows per order → dataset will be order-item level

6. REVENUE DEFINITION (confirmed):
   revenue = price + freight_value  (per item row)
""")


=== KEY FINDINGS FROM INSPECTION ===

1. ORDERS TABLE:
   - Contains ~100K orders with multiple date columns
   - order_delivered_customer_date has nulls (non-delivered orders)
   - We will filter to 'delivered' status for delivery analysis

2. ITEMS TABLE:
   - One-to-many with orders (some orders have multiple items)
   - price and freight_value are per item, not per order

3. PAYMENTS TABLE:
   - One-to-many with orders (installments create multiple rows)
   - Must aggregate by order_id before joining to items

4. REVIEWS TABLE:
   - Some orders may have duplicate reviews
   - Will use latest review or average — decide in cleaning step

5. JOIN RISK:
   - payments has multiple rows per order → must aggregate FIRST
   - items has multiple rows per order → dataset will be order-item level

6. REVENUE DEFINITION (confirmed):
   revenue = price + freight_value  (per item row)

